# 01 — EDA: internal job postings + employee-profile corpus

**Project:** AI Career Copilot (H09) · **Author:** Asad Kamran · MADS, University of Michigan · Dubai HR

Two parquet artefacts: 1,000 internal job postings and 2,000 employee profiles, both with skill-overlap structure that the TF-IDF retriever can exploit. The production stack slots in a LangGraph agent loop with tool calls; the v0.1 demo deliberately uses a TF-IDF + templated answer generator (`docs/03_methodology.md`).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from career_copilot.data import (
    PROCESSED, DEPTS, ASPIRATIONS, load_all, make_training_artifacts,
)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

In [ ]:
if (PROCESSED / 'postings.parquet').exists():
    postings, profiles = load_all()
else:
    postings, profiles = make_training_artifacts()
postings.shape, profiles.shape

## 1. Postings per department

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
postings['dept'].value_counts().plot(kind='bar', ax=ax, color='#3a7ca5')
ax.set_title('# postings per dept')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

## 2. Posting level distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
sns.countplot(data=postings, x='level', ax=ax, color='#9c6644')
ax.set_title('Posting level distribution')
plt.tight_layout(); plt.show()

## 3. Required-skills count per posting

In [ ]:
n_skills = postings['required_skills'].fillna('').apply(lambda s: len([x for x in s.split(',') if x.strip()]))
fig, ax = plt.subplots(figsize=(7, 3.4))
sns.histplot(n_skills, bins=20, ax=ax, color='#4a7c59')
ax.set_title('Required-skills count per posting')
plt.tight_layout(); plt.show()

## 4. Profiles per dept × tenure

In [ ]:
profiles['tenure_bucket'] = pd.cut(profiles['tenure_yrs'], bins=[0, 2, 5, 10, 30], labels=['<2y', '2-5y', '5-10y', '10y+'])
ct = profiles.groupby(['current_dept', 'tenure_bucket']).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(ct, annot=True, fmt='d', cmap='YlGnBu', ax=ax)
ax.set_title('Profiles per dept × tenure bucket')
plt.tight_layout(); plt.show()

## 5. Aspiration distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
profiles['aspiration'].value_counts().plot(kind='barh', ax=ax, color='#3a7ca5')
ax.set_title('Aspiration phrases across the profile corpus')
plt.tight_layout(); plt.show()

## 6. Cross-dept target structure

The synthetic generator chooses the `target_job_id` 80% within the employee's dept and 20% cross-dept. We confirm the marginal split.

In [ ]:
j_to_dept = postings.set_index('job_id')['dept'].to_dict()
profiles['target_dept'] = profiles['target_job_id'].map(j_to_dept)
cross = (profiles['target_dept'] != profiles['current_dept']).mean()
print(f'cross-dept target share: {cross:.2%}')

## 7. Hypotheses for `03_model.ipynb`

1. **TF-IDF retrieval over `posting_text` will recover the dept-aligned target_job_id within top-5 for the majority of profiles.** Cross-dept aspirations will be the harder slice.
2. **Profile + query blending lifts the cross-dept slice** because the aspiration phrase carries cross-dept signal the profile alone lacks.
3. **Templated answer generation suffices** to demonstrate the LangGraph slot — the production swap is one component.